<a href="https://colab.research.google.com/github/sittana-afifi/KTT-Fellow-Hackathon---G1---T2.1-Compressed-Crop-Disease-Classifier/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import io
import torch
import torch.nn as nn
import uvicorn
import threading
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import Response
from torchvision import models, transforms
from PIL import Image

# --- 1. MODEL & PREPROCESSING SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Architecture matching your trained hinga_model.pth
model = models.mobilenet_v3_small(weights=None)
model.classifier = nn.Sequential(
    nn.Linear(576, 1024),
    nn.Hardswish(inplace=True),
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(1024, 5)
)

# Load weights
model_path = 'models/hinga_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    print("✅ Weights loaded. Accuracy tracking enabled.")
else:
    print("⚠️ Weights not found. Using initialized model.")

class_names = ['Bean Spot', 'Cassava Mosaic', 'Maize Blight', 'Healthy Maize', 'Maize Rust']
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. FASTAPI & DATABASE ---
app = FastAPI()
# This dictionary stores the "Actual Result" string including the Accuracy %
ussd_results_db = {}

@app.post("/predict")
async def predict(phoneNumber: str = Form(...), file: UploadFile = File(...)):
    try:
        # Process Image
        data = await file.read()
        image = Image.open(io.BytesIO(data)).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(device)

        # Inference & Accuracy Calculation
        with torch.no_grad():
            output = model(input_tensor)
            prob = torch.nn.functional.softmax(output, dim=1)
            conf, pred_idx = torch.max(prob, 1)

        disease = class_names[pred_idx]
        accuracy = conf.item() * 100  # Convert to percentage

        # Store a formatted string for the USSD retrieval
        result_entry = f"{disease} (Confidence: {accuracy:.1f}%)"
        ussd_results_db[phoneNumber] = result_entry

        return {
            "status": "Success",
            "prediction": disease,
            "accuracy": f"{accuracy:.1f}%",
            "ussd_preview": f"Diagnosis: {disease}\nAccuracy: {accuracy:.1f}%"
        }
    except Exception as e:
        return {"error": str(e)}

@app.post("/ussd")
async def ussd_handler(phoneNumber: str = Form(...), text: str = Form("")):
    # Handle USSD Menu logic
    if text == "" or text == "0":
        response = "CON Welcome to Hinga-AI\n1. Check Latest Diagnosis\n2. Farming Tips"
    elif text == "1":
        # Pull the actual diagnosis + accuracy stored in Step 1
        result = ussd_results_db.get(phoneNumber, "No recent scan found.")
        response = f"END Your Result:\n{result}"
    else:
        response = "END Session closed."

    return Response(content=response, media_type="text/plain")

# --- 3. SERVER MANAGEMENT ---
# Kill any existing server on port 8000
!fuser -k 8000/tcp

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()
print("🚀 Server started. Accuracy will now appear in USSD responses.")

✅ Weights loaded. Accuracy tracking enabled.
🚀 Server started. Accuracy will now appear in USSD responses.


In [ ]:
import requests
url = "http://127.0.0.1:8000"
phone = "+250780000000"
img_path = '/content/ktt/data/mini_plant_set/maize_rust/49339.jpg'

# 1. Prediction (Calculates accuracy)
with open(img_path, 'rb') as f:
    requests.post(f"{url}/predict", data={'phoneNumber': phone}, files={'file': f})

# 2. USSD (Retrieves accuracy)
resp = requests.post(f"{url}/ussd", data={'phoneNumber': phone, 'text': '1'})

print("--- FARMER VIEW ---")
print(resp.text)

INFO:     127.0.0.1:41400 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:41414 - "POST /ussd HTTP/1.1" 200 OK
--- FARMER VIEW ---
END Your Result:
Maize Rust (Confidence: 99.7%)
